# Import / Lib

In [2]:
!pip install drain3

from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig
import pandas as pd
import numpy as np

  Preparing metadata (setup.py) ... done
  Created wheel for drain3: filename=drain3-0.9.11-py3-none-any.whl size=23998 sha256=058d1012b524f611c56361553cf8f8ae97602f2d73a6bf401a40a45217f480ff
  Stored in directory: /root/.cache/pip/wheels/3f/d1/46/58e1747b3d77c4990f838e1c1f610f5aab1a21889cc9bff5c2
Successfully built drain3
  Attempting uninstall: jsonpickle
    Found existing installation: jsonpickle 4.1.1
    Uninstalling jsonpickle-4.1.1:
      Successfully uninstalled jsonpickle-4.1.1
  Attempting uninstall: cachetools
    Found existing installation: cachetools 6.2.6
    Uninstalling cachetools-6.2.6:
      Successfully uninstalled cachetools-6.2.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyiceberg 0.11.1 requires cachetools<7.0,>=5.5, but you have cachetools 4.2.1 which is incompatible.


# Config / Global

In [3]:
LOGFILE = 'BGL.log'

In [4]:
log_lines = list()
with open(LOGFILE, 'r') as f:
    for line in f:
        log_lines.append(line)
print(f"Total: {len(log_lines)} lines")
log_lines[:10]

Total: 1706057 lines


['- 1117838570 2005.06.03 R02-M1-N0-C:J12-U11 2005-06-03-15.42.50.363779 R02-M1-N0-C:J12-U11 RAS KERNEL INFO instruction cache parity error corrected\n',
 '- 1117838570 2005.06.03 R02-M1-N0-C:J12-U11 2005-06-03-15.42.50.527847 R02-M1-N0-C:J12-U11 RAS KERNEL INFO instruction cache parity error corrected\n',
 '- 1117838570 2005.06.03 R02-M1-N0-C:J12-U11 2005-06-03-15.42.50.675872 R02-M1-N0-C:J12-U11 RAS KERNEL INFO instruction cache parity error corrected\n',
 '- 1117838570 2005.06.03 R02-M1-N0-C:J12-U11 2005-06-03-15.42.50.823719 R02-M1-N0-C:J12-U11 RAS KERNEL INFO instruction cache parity error corrected\n',
 '- 1117838570 2005.06.03 R02-M1-N0-C:J12-U11 2005-06-03-15.42.50.982731 R02-M1-N0-C:J12-U11 RAS KERNEL INFO instruction cache parity error corrected\n',
 '- 1117838571 2005.06.03 R02-M1-N0-C:J12-U11 2005-06-03-15.42.51.131467 R02-M1-N0-C:J12-U11 RAS KERNEL INFO instruction cache parity error corrected\n',
 '- 1117838571 2005.06.03 R02-M1-N0-C:J12-U11 2005-06-03-15.42.51.293532 R02

In [42]:
def build_miner(sim_th=0.4, drain_depth=4):
  config = TemplateMinerConfig()
  config.drain_sim_th = sim_th
  config.drain_depth = drain_depth

  miner = TemplateMiner(config=config)

  return miner

def parser(log_lines, sim_th=0.4, drain_depth=4):
  miner = build_miner(sim_th, drain_depth)

  log_parsed = []
  templates = []

  for line in log_lines:
    result = miner.add_log_message(line)
    data = {
        'template_id': result['cluster_id'],
        'template': result['template_mined'],
        'log': line,
        'timestamp': int(line.split(' ')[1]),
        'truth_label': 1 if line.split(' ')[0] == '-' else -1
    }
    log_parsed.append(data)


  for cluster in miner.drain.clusters:
    data = {
        'id': cluster.cluster_id,
        'count': cluster.size,
        'template': cluster.get_template()
    }
    templates.append(data)

  return log_parsed, templates, miner


# Tuning SIMTH

In [11]:
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]

for th in thresholds:
  log_parsed, templates, miner = parser(log_lines, sim_th=th)

  print(f"Chỉ số Threshold = {th}: {len(templates)} templates")

Chỉ số Threshold = 0.1: 67 templates
Chỉ số Threshold = 0.2: 95 templates
Chỉ số Threshold = 0.3: 157 templates
Chỉ số Threshold = 0.4: 253 templates
Chỉ số Threshold = 0.5: 545 templates


In [43]:
log_parsed, templates, miner = parser(log_lines, sim_th=0.3) # Số lượng template ở mức trung bình, không quá lớn. Tốc độ build miner cũng vùa phải không quá lâu
print(f"Tổng số template: {len(templates)}")
for template in templates:
  print(template)

Tổng số template: 157
{'id': 1, 'count': 64933, 'template': '- <*> <*> <*> <*> <*> RAS KERNEL INFO instruction cache parity error corrected'}
{'id': 2, 'count': 388, 'template': '- <*> <*> <*> <*> <*> RAS LINKCARD INFO MidplaneSwitchController performing bit sparing on <*> bit <*>'}
{'id': 3, 'count': 767777, 'template': '- <*> <*> <*> <*> <*> RAS KERNEL INFO <*> <*>'}
{'id': 4, 'count': 4413, 'template': '- <*> <*> <*> <*> <*> RAS KERNEL INFO <*> ddr errors(s) detected and corrected on rank 0, symbol <*> bit <*>'}
{'id': 5, 'count': 1481, 'template': '- <*> <*> <*> <*> <*> RAS KERNEL INFO <*> <*> <*> error(s) (dcr <*> detected and corrected'}
{'id': 6, 'count': 651, 'template': '- <*> 2005.06.03 <*> <*> <*> RAS KERNEL INFO <*> <*> <*> <*> <*> <*> <*>'}
{'id': 7, 'count': 4, 'template': '- <*> <*> R16-M1-N2-C:J17-U01 <*> R16-M1-N2-C:J17-U01 RAS KERNEL INFO <*> <*> <*> <*> <*> <*> <*> <*>'}
{'id': 8, 'count': 3408, 'template': '- <*> <*> <*> <*> <*> RAS KERNEL INFO total of <*> ddr erro

In [29]:
top_templates = sorted(templates, key=lambda x: x['count'], reverse=True)[:10]
pd.DataFrame(top_templates).to_csv('top_templates.csv', index=False)

In [33]:
def build_time_series(log_parsed):
  df = pd.DataFrame(log_parsed)
  time_series_df = df.groupby(['timestamp', 'template_id']).size().unstack(fill_value=0)
  df_flat = time_series_df.reset_index()
  df_flat.columns.name = None
  df_flat = df_flat.rename(columns={'timestamp': 'timestamp'})
  return df_flat

time_series = build_time_series(log_parsed)
time_series = time_series.set_index('timestamp')
time_series

,1,2,3,4,5,6,7,8,9,10,...,148,149,150,151,152,153,154,155,156,157
timestamp,,,,,,,,,,,,,,,,,,,,,
1117838570,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1117838571,6,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1117838572,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1117838573,6,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1117838574,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1120934811,0,0,33,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1120934812,0,0,33,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1120934813,0,0,33,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [32]:
true_label_df = pd.DataFrame(log_parsed).groupby('timestamp')['truth_label'].min().reset_index().rename(columns={'truth_label': 'label'})
true_label_df

,timestamp,label
0,1117838570,1
1,1117838571,1
2,1117838572,1
3,1117838573,1
4,1117838574,1
...,...,...
86852,1120934811,1
86853,1120934812,1
86854,1120934813,1
86855,1120934814,1


In [34]:
true_label_df[true_label_df['label'] == -1]

,timestamp,label
1859,1117869872,-1
1860,1117869873,-1
1861,1117869874,-1
1862,1117869875,-1
1863,1117869876,-1
...,...,...
83218,1120912249,-1
83219,1120912250,-1
83220,1120912251,-1
83231,1120913153,-1


# Isolation Forest + Tuning Contamination

In [35]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score, f1_score

def build_if(series, contamination=0.3):
  X = series.values
  model = IsolationForest(contamination=contamination, n_estimators=100, random_state=42, n_jobs=-1)
  model.fit(X)

  return model

def calc_metrics_sklearn(true_labels: list, pred_labels: list):
    # pos_label=-1 chỉ định tính toán tập trung vào nhãn -1
    precision = precision_score(true_labels, pred_labels, pos_label=-1, zero_division=0)
    recall = recall_score(true_labels, pred_labels, pos_label=-1, zero_division=0)
    f1 = f1_score(true_labels, pred_labels, pos_label=-1, zero_division=0)

    return {
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1
    }

contaminations = [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45]

for contamination in contaminations:
  model = build_if(time_series, contamination=contamination)
  pred_labels = model.predict(time_series)
  true_labels = true_label_df['label'].values
  metrics = calc_metrics_sklearn(true_labels, pred_labels)
  print(f"Contamination = {contamination}: {metrics}")



Contamination = 0.1: {'Precision': 0.17049027599860253, 'Recall': 0.05707602339181286, 'F1-Score': 0.0855215118147034}
Contamination = 0.15: {'Precision': 0.21111810033024062, 'Recall': 0.10467836257309941, 'F1-Score': 0.1399603836530442}
Contamination = 0.2: {'Precision': 0.19546873193850423, 'Recall': 0.13185185185185186, 'F1-Score': 0.15747811510523374}
Contamination = 0.25: {'Precision': 0.17086662569145666, 'Recall': 0.14089668615984405, 'F1-Score': 0.15444114441999102}
Contamination = 0.3: {'Precision': 0.1648474133292336, 'Recall': 0.16721247563352826, 'F1-Score': 0.16602152202523807}
Contamination = 0.35: {'Precision': 0.1489767426022406, 'Recall': 0.1705653021442495, 'F1-Score': 0.15904175073159205}
Contamination = 0.4: {'Precision': 0.1489767426022406, 'Recall': 0.1705653021442495, 'F1-Score': 0.15904175073159205}
Contamination = 0.45: {'Precision': 0.1249250449730162, 'Recall': 0.1705653021442495, 'F1-Score': 0.14422046776878575}


# Embedding

In [41]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import DBSCAN

# TF-IDF
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform([template['template'] for template in templates])
sim_matrix = cosine_similarity(tfidf_matrix)

# Nhóm cluster
distance_matrix = np.clip(1 - sim_matrix, 0, 1)

for esp in np.arange(0.0, 0.5 + 0.01, 0.05):
  if esp == 0.0:
    continue
  dbscan = DBSCAN(eps=esp, min_samples=2, metric='precomputed')
  cluster_labels = dbscan.fit_predict(distance_matrix)

  for template, label in zip(templates, cluster_labels):
      template['cluster_id'] = label

  print(f"cluster label cho esp {esp}: {cluster_labels}")

cluster label cho esp 0.05: [-1 -1  0 -1 -1  1 -1 -1 -1  1 -1 -1 -1 -1 -1 -1  2 -1  3 -1  2  2  2 -1
 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1  3  4  4  4 -1 -1 -1 -1 -1 -1 -1
 -1 -1  5 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1  6  6 -1
 -1 -1 -1 -1  7  7  7 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1  8 -1  8
  9  9  9 -1 -1 -1 -1  0  0 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1  5 -1 -1 -1
 -1 -1 -1 -1 10 10 10 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
 11 11 -1 11 11 -1 -1 -1  2 -1 -1 -1 -1]
cluster label cho esp 0.1: [-1 -1  0 -1 -1  1 -1 -1 -1  1 -1 -1 -1 -1 -1 -1  2 -1  3 -1  2  2  2 -1
 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1  3  4  4  4 -1 -1 -1 -1  5 -1 -1
 -1 -1  6 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1  7  7 -1
 -1 -1 -1 -1  8  8  8 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1  9 -1  9
 10 10 10  5 -1 -1 -1  0  0 -1 -1 -1 -1 -1 -1 -1 -1 -1 11 11  6 -1 -1 -1
 -1 -1 -1 -1 12 12 12  6 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
 13 13 -1 13 13 -1 -1 -1  2 

# Detect New Log

In [50]:
new_logs = ["Detect a virus from IP 10.2.39.122", "Reject a virus from IP 10.2.21.122"]

for new_log in new_logs:
  result = miner.add_log_message(new_log)

  if result["change_type"] == "cluster_created":
      print("\n[CẢNH BÁO] ĐÃ PHÁT HIỆN TEMPLATE LOG MỚI XUẤT HIỆN!")
      print(f"ID cụm mới: {result['cluster_id']}")
      print(f"Nội dung Template: {result['template_mined']}")
      print(f"Log thô kích hoạt: {new_log}\n")

  elif result["change_type"] == "cluster_template_changed":
      print(f"[Cập nhật] Template ID {result['cluster_id']} thay đổi cấu trúc sang dạng: {result['template_mined']}")
  else:
      print("Không có sự thay đổi nào được phát hiện.")

Không có sự thay đổi nào được phát hiện.

[CẢNH BÁO] ĐÃ PHÁT HIỆN TEMPLATE LOG MỚI XUẤT HIỆN!
ID cụm mới: 161
Nội dung Template: Reject a virus from IP 10.2.21.122
Log thô kích hoạt: Reject a virus from IP 10.2.21.122

